# 因子构造 
**AED**因子构建    
取agent的行动 $action_{i,s,t}$  
其中，$i$ 为agent的编号，$s$ 为agent的行动，$t$ 为时间窗口     
对于每一个证券-时间窗口，求所有agent的行动的平均值与标准差的比值，作为AED因子。     

如果所有agent都做出了高的决策，则AED大，如果所有agent都做出了低的决策，则AED小，如果agent的决策差异大，则AED也小。   

$$
AED_{s,t} = \frac{\sum_{i=1}^{n} action_{i,s,t}}{n} / \sqrt{\frac{\sum_{i=1}^{n} (action_{i,s,t} - \bar{action_{s,t}})^2}{n}}
$$

按照语义，由于训练是 拆分证券 + 多次训练 + 多次保存，一般来说，一次任务中会包括多个目录，多个jsonl。一般来说，一个task_prefix代表着同一个实验，其数据共用。   

## 导入库

In [6]:
import json  
import os
import re
import polars as pl
from pathlib import Path
import warnings

## 超参数

In [7]:
TASK_ID_PREFIX = 'test'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

## 读取数据
数据的保存路径为：    

```
DATA_BASE_DIR  
|- TASK_ID1
|   |- Node1
|   |   |- performance_and_record_0.json   
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- Node2
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- ...
|- TASK_ID2
|   |- Node1
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
```  

其中，task_id的构成为 时间_中缀_uuid，例如：20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57  
指定中缀，选取所有中缀一样的task_id，得到所有node的路径  

In [8]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

node_paths


['/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988592_15963',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988574_30803',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996658_s16',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996652_s30978',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996659_s3504',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996658_s15936',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996659_s30978',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_t

In [9]:
# 获取其下所有performance_and_record_*.jsonl文件的路径
jsonl_paths = []
for node_path in node_paths:
    all_files = os.listdir(node_path)
    perf_and_rewa_jsonl_files = [file for file in all_files if file.endswith('.jsonl') and 'performance_and_reward_' in file]
    jsonl_paths.extend(
        os.path.join(node_path, file)
        for file in perf_and_rewa_jsonl_files
    )
jsonl_paths

['/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988592_15963/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988592_15963/performance_and_reward_1.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988574_30803/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771988574_30803/performance_and_reward_1.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996658_s16/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260225_1108_test_2d2e3cb6-322a-4246-ad52-00a36d8c5970/node_1771996658_s16/performance_and_reward_1.jsonl',
 '/home/frank/files/

先用scan获取所有的jsonl  
使用polars的concat方法，合并所有json，字段由performance_and_record约定好  
展示测试数据  

In [10]:
lazy_frames = [pl.scan_ndjson(f) for f in jsonl_paths]
lf = pl.concat(lazy_frames)   # 得到 LazyFrame
lf.head().collect(engine = 'auto')


year,month,portfolio,data
i64,i64,list[str],struct[4]
2023,1,"[""301150""]","{[1.0, 0.0],[-0.0461, -0.0754, … -0.0],[-1.3752, -7.5376, … -0.8998],-0.053}"
2023,1,"[""001269""]","{[0.7865, 0.2135],[0.013, -0.0065, … 0.7482],[-0.3709, -0.6521, … 0.7957],0.013}"
2023,1,"[""001283""]","{[0.0762, 0.9238],[0.0015, -0.0312, … 0.3885],[-0.5677, -3.1185, … -0.0194],0.0003}"
2023,1,"[""001238""]","{[0.4332, 0.5668],[0.0172, -0.0086, … 0.9871],[-0.2998, -0.8614, … 1.3371],0.0171}"
2023,1,"[""688380""]","{[0.1513, 0.8487],[-0.0029, -0.0117, … 0.6131],[-0.6414, -1.1749, … 0.4895],-0.0031}"


## 处理数据  
加载的lf，其结构如上所示，需要进行解析   

>- 1.基线回归为单证券，因此，需要验证portfolio中列表长度，如果有长度大于1的列表，需要警告，并且取[0]  
>- 2. data比较复杂，由多项fields组成，包括decision_weights, performance, normalized_performance, reward； 其中，如果是多证券，decision_weights, 是list结构，需要判断是否有长度大于1的列表，如果有，需要警告，并且取[0]; performance 和 normalized_performance是固定len=5的list，对应字段return, -vol, sharp, -maxdrawdwon, devisification，需要展开为对应字段   

In [11]:
# 展示fields
df_demo = lf.head().collect()
df_demo.with_columns(pl.col('data').struct.unnest()).select(pl.all().exclude('data'))

year,month,portfolio,decision_weights,performance,normalized_performance,reward
i64,i64,list[str],list[f64],list[f64],list[f64],f64
2023,1,"[""301150""]","[1.0, 0.0]","[-0.0461, -0.0754, … -0.0]","[-1.3752, -7.5376, … -0.8998]",-0.053
2023,1,"[""001269""]","[0.7865, 0.2135]","[0.013, -0.0065, … 0.7482]","[-0.3709, -0.6521, … 0.7957]",0.013
2023,1,"[""001283""]","[0.0762, 0.9238]","[0.0015, -0.0312, … 0.3885]","[-0.5677, -3.1185, … -0.0194]",0.0003
2023,1,"[""001238""]","[0.4332, 0.5668]","[0.0172, -0.0086, … 0.9871]","[-0.2998, -0.8614, … 1.3371]",0.0171
2023,1,"[""688380""]","[0.1513, 0.8487]","[-0.0029, -0.0117, … 0.6131]","[-0.6414, -1.1749, … 0.4895]",-0.0031


In [12]:
# 定义字段
perf_fields = ["return", "neg_vol", "sharp", "neg_maxdrawdown", "diversification"]

# 1. 在 LazyFrame 上完成 unnest，再 collect（单次扫描，避免先 collect 再在 Python 里 unnest）
lf_unnested = lf.with_columns(pl.col("data").struct.unnest()).drop("data")
df_flat = lf_unnested.collect()  # 需要子集时可改为 lf_unnested.head(n).collect()

# 2. portfolio：检查是否有多证券，有则告警并只保留 [0]
if (df_flat["portfolio"].list.len() > 1).any():
    warnings.warn("发现 portfolio 列表长度 > 1（多证券），已取 [0] 作为单证券基线。")
df_flat = df_flat.with_columns(pl.col("portfolio").list.get(0).alias("portfolio"))

# 3. decision_weights：多证券时告警并取 [0]
if (df_flat["decision_weights"].list.len() > 1).any():
    warnings.warn("发现 decision_weights 列表长度 > 1，已取 [0]。")
df_flat = df_flat.with_columns(pl.col("decision_weights").list.get(0).alias("decision_weights"))

# 4. performance / normalized_performance：固定 len=5，展开为多列（return, -vol, sharp, -maxdrawdown, diversification）
df_flat = df_flat.with_columns(
    pl.col("performance").list.to_struct(fields=[f"{f}" for f in perf_fields]).struct.unnest()
).drop("performance")
df_flat = df_flat.with_columns(
    pl.col("normalized_performance").list.to_struct(fields=[f"normalized_{f}" for f in perf_fields]).struct.unnest()
).drop("normalized_performance")
# reward 已是标量，无需处理

/tmp/ipykernel_16929/3039891457.py:15: UserWarning: 发现 decision_weights 列表长度 > 1，已取 [0]。
  warnings.warn("发现 decision_weights 列表长度 > 1，已取 [0]。")


In [13]:
df_flat.head()

year,month,portfolio,decision_weights,reward,return,neg_vol,sharp,neg_maxdrawdown,diversification,normalized_return,normalized_neg_vol,normalized_sharp,normalized_neg_maxdrawdown,normalized_diversification
i64,i64,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023,1,"""301150""",1.0,-0.053,-0.0461,-0.0754,0.3884,-0.0461,-0.0,-1.3752,-7.5376,38.8388,-4.61,-0.8998
2023,1,"""001269""",0.7865,0.013,0.013,-0.0065,0.9998,-0.0,0.7482,-0.3709,-0.6521,99.9847,-0.0,0.7957
2023,1,"""001283""",0.0762,0.0003,0.0015,-0.0312,1.0465,-0.0,0.3885,-0.5677,-3.1185,104.6479,-0.0,-0.0194
2023,1,"""001238""",0.4332,0.0171,0.0172,-0.0086,0.9999,-0.0,0.9871,-0.2998,-0.8614,99.9884,-0.0,1.3371
2023,1,"""688380""",0.1513,-0.0031,-0.0029,-0.0117,0.7539,-0.0029,0.6131,-0.6414,-1.1749,75.3926,-0.289,0.4895


In [14]:
# 转为lf，方便后续处理
lf_flat = df_flat.lazy()

## 构建因子
为了减少处理难度，先筛选需要的列  

- year   
- month    
- portfolio    
- decision_weights     
- return (表示year-month下一个月的收益)   

In [15]:
needed_cols = ['year','month','portfolio','decision_weights','return']
filtered_lf = lf_flat.select(
    pl.col(needed_cols)
)
filtered_lf.head().collect()

year,month,portfolio,decision_weights,return
i64,i64,str,f64,f64
2023,1,"""301150""",1.0,-0.0461
2023,1,"""001269""",0.7865,0.013
2023,1,"""001283""",0.0762,0.0015
2023,1,"""001238""",0.4332,0.0172
2023,1,"""688380""",0.1513,-0.0029


按照AED因子的定义，构建因子，按照year,month,portfolio进行分组，求每一个组decision_weights的 平均值 / 标准差，作为AED因子  

In [16]:
aed_lf = filtered_lf.group_by(['year','month','portfolio']).agg(
    (pl.col('decision_weights').mean() / pl.col('decision_weights').std()).alias('AED'),
    pl.col('return').first().alias('return')
)
aed_lf.head().collect()

year,month,portfolio,AED,return
i64,i64,str,f64,f64
2023,5,"""688244""",0.928687,-0.1351
2023,11,"""688401""",1.962685,-0.0002
2023,12,"""301309""",0.833357,-0.1939
2023,5,"""301226""",18.41939,0.0482
2023,3,"""688223""",0.805795,-0.1025


最后，将year,month合并为date  

In [17]:
aed_lf = aed_lf.with_columns(
    pl.date(pl.col('year'), pl.col('month'), 1).alias('date')
).drop(['year','month']).select(['date','portfolio','AED','return'])
aed_lf.head().collect()

date,portfolio,AED,return
date,str,f64,f64
2023-01-01,"""301338""",1.885814,-0.0225
2023-10-01,"""688291""",2.12132,0.0494
2023-04-01,"""688153""",1.954444,0.0217
2023-11-01,"""688279""",1.471006,0.0314
2023-09-01,"""301266""",1.054127,-0.0046


## 保存数据  
保存数据为parquet文件  

In [18]:
if SAVE:
    dir_path = Path(SAVE_BASELINE_REG_DIR)
    dir_path.mkdir(parents=True, exist_ok=True)
    aed_lf.collect().write_parquet(os.path.join(SAVE_BASELINE_REG_DIR, f'基准回归-AED因子.parquet'))
    print(f'保存成功，路径为:{os.path.join(SAVE_BASELINE_REG_DIR, f"基准回归-AED因子.parquet")}')
else:
    print('请设置SAVE=True，以保存数据')

保存成功，路径为:/home/frank/files/programs/GraduationThesis/empirical/test/baseline_reg/基准回归-AED因子.parquet
